# Kredi Kartı Temerrüt Tahmini — Baseline Pipeline

**Araştırma Sorusu:** Demografik özellikler ve ödeme geçmişi, bir müşterinin bir sonraki ay ödeme yapıp yapamayacağını tahmin edebilir mi? Farklı sınıflandırıcılar bu problemde nasıl karşılaştırılır?

**Dataset:** UCI Default of Credit Card Clients  
- 30.000 kayıt, 23 özellik, 1 hedef değişken (`default.payment.next.month`)  
- Hedef: 0 = ödeme yapacak, 1 = temerrüde düşecek (~22% pozitif)

**Pipeline sırası:**
1. EDA (Exploratory Data Analysis) — veriyi tanı
2. Preprocessing — veriyi temizle ve hazırla
3. Train/Test Split — eğitim ve test ayrımı
4. Model Eğitimi — 4 farklı algoritma
5. Değerlendirme — metrikler ve karşılaştırma
6. Görselleştirme — ROC eğrisi, feature importance

## Adım 1 — EDA (Veriyi Tanıma)

Veriyi ilk kez görüyoruz: boyutu, ilk satırlar, temel istatistikler, eksik değer kontrolü, hedef dağılımı.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from data_loader import load_dataset

X, y, _variables = load_dataset()

# Veriyi düzleştir: tek bir DataFrame yap
df = X.copy()
df['default'] = y.values  # hedef sütunu ekle

# --- Temel bilgiler ---
print("=== Boyut ===")
print(f"Satır: {df.shape[0]:,}  |  Sütun: {df.shape[1]}")

print("\n=== İlk 5 satır ===")
display(df.head())

print("\n=== Temel istatistikler ===")
display(df.describe().round(2))

print("\n=== Eksik değer sayısı ===")
print(df.isnull().sum().sum(), "eksik değer bulundu.")

print("\n=== Hedef değişken dağılımı ===")
counts = df['default'].value_counts()
pct = df['default'].value_counts(normalize=True) * 100
print(pd.DataFrame({'Adet': counts, 'Yüzde (%)': pct.round(1)}))

In [ ]:
# --- Görsel 1: Hedef dağılımı (class imbalance) ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Hedef Değişken Dağılımı')
axes[0].set_xlabel('default (0=Hayır, 1=Evet)')
axes[0].set_ylabel('Adet')
axes[0].set_xticklabels(['Temerrüt Yok (0)', 'Temerrüt (1)'], rotation=0)
for bar, count in zip(axes[0].patches, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{count:,}', ha='center', va='bottom', fontweight='bold')

# Korelasyon ısı haritası (ilk 10 özellik, okunabilirlik için)
top_cols = list(X.columns[:10]) + ['default']
corr = df[top_cols].corr()
sns.heatmap(corr, ax=axes[1], annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, annot_kws={'size': 7})
axes[1].set_title('Korelasyon Isı Haritası (ilk 10 özellik)')

plt.tight_layout()
plt.show()
print("Not: ~78% negatif, ~22% pozitif → class imbalance var.")


## Adım 2 — Preprocessing

- `EDUCATION` ve `MARRIAGE` sütunlarındaki hatalı kodlar düzeltiliyor (belgelenmemiş değerler → "Other")
- Tüm sayısal özellikler **StandardScaler** ile normalize ediliyor (k-NN ve Naive Bayes için önemli)
- Class imbalance için modellere `class_weight='balanced'` parametresi verilecek

In [ ]:
from sklearn.preprocessing import StandardScaler

# --- Kopyayı al (orijinal X'i bozmamak için) ---
X_clean = X.copy()

# =====================================================================
# ÖNCE: Ham verideki gerçek değerleri say —
# hangi değerler var, kaç kez geçiyor?
# Dokümantasyon ne diyorsa desin, veriyi biz kontrol ediyoruz.
# =====================================================================
print("=== X3 (EDUCATION) — ham değer dağılımı ===")
print(X_clean['X3'].value_counts().sort_index())
print("\nDokümantasyona göre geçerli değerler: 1=Yüksek Lisans, 2=Üniversite, 3=Lise, 4=Diğer")
print("Yukarıda bunların dışında değer görünüyorsa → belgelenmemiş → 'Diğer' (4) yapacağız\n")

print("=== X4 (MARRIAGE) — ham değer dağılımı ===")
print(X_clean['X4'].value_counts().sort_index())
print("\nDokümantasyona göre geçerli değerler: 1=Evli, 2=Bekar, 3=Diğer")
print("0 görünüyorsa → belgelenmemiş → 'Diğer' (3) yapacağız\n")

# =====================================================================
# SONRA: Gördüklerimize göre düzeltelim
# Geçersiz değerleri "Diğer" kategorisine çekiyoruz.
# replace() dict'i: {"eski_değer": "yeni_değer", ...}
# Listede olmayan değerlere dokunmaz → bunu da kontrol edeceğiz.
# =====================================================================
gecerli_X3 = {1, 2, 3, 4}
gecerli_X4 = {1, 2, 3}

gecersiz_X3 = set(X_clean['X3'].unique()) - gecerli_X3
gecersiz_X4 = set(X_clean['X4'].unique()) - gecerli_X4
print(f"X3'te geçersiz değerler (düzeltilecek): {sorted(gecersiz_X3)}")
print(f"X4'te geçersiz değerler (düzeltilecek): {sorted(gecersiz_X4)}")

# Yukarıda ne gördüysek replace dict'ini buna göre doldur
X_clean['X3'] = X_clean['X3'].replace({v: 4 for v in gecersiz_X3})
X_clean['X4'] = X_clean['X4'].replace({v: 3 for v in gecersiz_X4})

print("\n--- Düzeltme sonrası ---")
print("X3 unique:", sorted(X_clean['X3'].unique()), "  ← sadece 1,2,3,4 kalmalı")
print("X4 unique:", sorted(X_clean['X4'].unique()), "  ← sadece 1,2,3 kalmalı")

# --- Hedef değişkeni düzleştir (1D array yap) ---
y_arr = y.values.ravel()  # shape (30000,)

# --- StandardScaler: mean=0, std=1 dönüşümü ---
# Neden? k-NN mesafe hesaplar → ölçek farkı varsa büyük sayılı sütunlar baskın olur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)

print(f"\nX_scaled shape: {X_scaled.shape}")
print(f"y_arr shape: {y_arr.shape}")
print(f"İlk özelliğin yeni mean ≈ {X_scaled[:, 0].mean():.4f}  (0'a yakın olmalı)")
print(f"İlk özelliğin yeni std  ≈ {X_scaled[:, 0].std():.4f}  (1'e yakın olmalı)")


## Adım 3 — Train / Test Split

Veriyi %80 eğitim, %20 test olarak bölüyoruz.  
`stratify=y_arr` → class dağılımının iki sette de aynı kalmasını sağlar (önemli!)  
`random_state=42` → aynı bölünmeyi tekrar elde etmek için sabit tohum

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_arr,
    test_size=0.20,       # %20 test
    random_state=42,      # tekrarlanabilirlik
    stratify=y_arr        # class oranlarını koru
)

print(f"Eğitim seti : {X_train.shape[0]:,} satır")
print(f"Test seti   : {X_test.shape[0]:,} satır")
print(f"\nTest setinde temerrüt oranı: {y_test.mean():.1%}  (orijinalle aynı olmalı)")


## Adım 4 — Modelleri Eğit

Topic 7'nin gerektirdiği 4 model:

| Model | Temel fikir |
|---|---|
| **Decision Tree** | Veriyi soru ağacıyla böler (if-else kuralları) |
| **Naïve Bayes** | Olasılık hesabıyla sınıflandırır, özellikler bağımsız varsayılır |
| **k-NN** | Yeni örneğe en yakın k komşunun sınıfına bakar |
| **Random Forest** | Yüzlerce Decision Tree'nin oy birliği (ensemble) |

`class_weight='balanced'` → modelin azınlık sınıfa (temerrüt=1) daha fazla dikkat etmesini sağlar

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# --- Model tanımları ---
models = {
    'Decision Tree': DecisionTreeClassifier(
        max_depth=10,           # çok derin ağaç ezberleme yapar (overfitting)
        class_weight='balanced',
        random_state=42
    ),
    'Naïve Bayes': GaussianNB(),   # NB class_weight desteklemiyor, ama küçük etki
    'k-NN (k=5)': KNeighborsClassifier(
        n_neighbors=5         # en yakın 5 komşuya bak
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,       # 100 ağaç
        max_depth=10,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1               # tüm CPU çekirdeklerini kullan (hız için)
    ),
}

# --- Hepsini eğit ---
print("Modeller eğitiliyor...")
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"  ✓ {name}")

print("\nTüm modeller eğitildi.")


## Adım 5 — Değerlendirme

Metrikler:
- **Accuracy**: genel doğruluk — class imbalance varsa yanıltıcı olabilir
- **Precision**: "temerrüt dedi" dediğinde gerçekten kaçı temerrüt?
- **Recall**: gerçek temerrütlerin kaçını yakaladı?
- **F1-score**: Precision ve Recall'un harmonik ortası
- **ROC-AUC**: eşik değerinden bağımsız genel ayrım gücü

Raporun en önemli tablosu bu karşılaştırma tablosu olacak.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix
)

results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    # predict_proba: olasılık tahmini (pozitif sınıf için) — ROC-AUC hesabı için lazım
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob),
    })

results_df = pd.DataFrame(results).set_index('Model')
print("=== Model Karşılaştırma Tablosu ===")
results_display = results_df.round(4)
display(results_display)



In [ ]:
# --- Confusion Matrix: 4 modeli yan yana göster ---
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred: 0', 'Pred: 1'],
                yticklabels=['True: 0', 'True: 1'])
    ax.set_title(name)
    ax.set_xlabel('Tahmin')
    ax.set_ylabel('Gerçek')

plt.suptitle('Confusion Matrix Karşılaştırması', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("Sol üst (TN) + Sağ alt (TP) = doğru tahminler.  Sağ üst (FP) + Sol alt (FN) = hatalar.")


## Adım 6 — ROC Eğrisi + Feature Importance

ROC eğrisi: her eşik değerinde True Positive Rate vs False Positive Rate.  
AUC (alan altındaki alan) ne kadar 1'e yakınsa model o kadar iyi ayrım yapıyor.  
Feature Importance: Random Forest'ın hangi sütunlara ne kadar güvendiği.

In [ ]:
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- ROC Eğrileri ---
colors = ['steelblue', 'darkorange', 'green', 'red']
for (name, model), color in zip(models.items(), colors):
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Rastgele Tahmin')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Eğrisi Karşılaştırması')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].grid(alpha=0.3)

# --- Feature Importance (Random Forest) ---
rf_model = models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=X_clean.columns)
top15 = importances.nlargest(15).sort_values()

top15.plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('Feature Importance (Random Forest, Top 15)')
axes[1].set_xlabel('Önem Skoru')

plt.tight_layout()
plt.show()


## Baseline Pipeline Tamamlandı ✓

### Bu notebook'ta yapılanlar:
1. **EDA** — veri boyutu, dağılım, korelasyon, class imbalance tespit
2. **Preprocessing** — hatalı kategori düzeltme, StandardScaler normalizasyonu
3. **Train/Test Split** — %80/%20, stratified
4. **4 Model eğitimi** — Decision Tree, Naïve Bayes, k-NN, Random Forest
5. **Değerlendirme** — Accuracy / Precision / Recall / F1 / ROC-AUC karşılaştırma tablosu
6. **Görselleştirme** — Confusion Matrix, ROC Eğrisi, Feature Importance

### Rapora taşınacak unsurlar:
- Karşılaştırma tablosu → Results bölümü
- Confusion Matrix figürü → Results bölümü  
- ROC eğrisi → Results bölümü
- Feature Importance → Discussion bölümü ("hangi özellikler belirleyici?")
- EDUCATION/MARRIAGE düzeltmesi → Methodology/Preprocessing açıklaması
- Class imbalance tartışması → Discussion ("neden accuracy yanıltıcı?")